In [1]:
%load_ext line_profiler
%load_ext memory_profiler

In [2]:
from ensembles import Config, Ensemble, EnsembleFormatError, EnsembleWriter
from decode_bin import decode_bin

PATH = "200x.000"

In [3]:
import gc
import traceback

# Save the original gc.collect method
_original_gc_collect = gc.collect

# Define a wrapper that logs when gc.collect is called
def wrapped_gc_collect(*args, **kwargs):
    print("\n⚠️ gc.collect() called!")
    traceback.print_stack(limit=5)
    return _original_gc_collect(*args, **kwargs)

# Replace gc.collect with the wrapper
gc.collect = wrapped_gc_collect

In [4]:
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()

decode_bin(PATH)

profiler.disable()
stats = pstats.Stats(profiler).sort_stats("cumulative")
stats.print_stats(40)

         144065188 function calls (144065113 primitive calls) in 98.582 seconds

   Ordered by: cumulative time
   List reduced from 321 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       24    1.823    0.076  210.368    8.765 /home/zac/miniforge3/lib/python3.12/asyncio/base_events.py:1909(_run_once)
      3/2    0.000    0.000  102.996   51.498 /home/zac/miniforge3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3541(run_code)
      3/2    0.000    0.000  102.996   51.498 {built-in method builtins.exec}
  3179399   36.805    0.000   64.813    0.000 /home/zac/GitHub/COBIAlab-ADCP/ensembles.py:102(from_bytes)
       23    2.738    0.119   42.544    1.850 /home/zac/miniforge3/lib/python3.12/selectors.py:451(select)
      102    3.620    0.035   39.101    0.383 {built-in method time.sleep}
      777    0.262    0.000   28.177    0.036 /home/zac/GitHub/COBIAlab-ADCP/ensembles.py:313(write_batch)
      777    1.162    

In [5]:
tstats = pstats.Stats(profiler).sort_stats("tottime")
tstats.print_stats(40)

         144065188 function calls (144065113 primitive calls) in 98.582 seconds

   Ordered by: internal time
   List reduced from 321 to 40 due to restriction <40>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  3179399   36.805    0.000   64.813    0.000 /home/zac/GitHub/COBIAlab-ADCP/ensembles.py:102(from_bytes)
     4662    9.287    0.002   10.390    0.002 /home/zac/miniforge3/lib/python3.12/site-packages/numpy/_core/shape_base.py:371(stack)
 15896995    8.583    0.000    8.583    0.000 {built-in method numpy.frombuffer}
    13209    6.246    0.000    6.526    0.000 /home/zac/miniforge3/lib/python3.12/site-packages/h5py/_hl/dataset.py:786(__getitem__)
    13209    4.147    0.000    4.895    0.000 /home/zac/miniforge3/lib/python3.12/site-packages/h5py/_hl/dataset.py:892(__setitem__)
 19076395    3.887    0.000    3.887    0.000 {built-in method numpy.zeros}
      102    3.620    0.035   39.101    0.383 {built-in method time.sleep}
  9538214    3.181    0.0

In [11]:
%lprun -f Ensemble.from_bytes decode_bin(PATH)

*** KeyboardInterrupt exception caught in code being profiled.

Timer unit: 1e-09 s

Total time: 0 s
File: /home/zac/GitHub/COBIAlab-ADCP/ensembles.py
Function: write_batch at line 305

Line #      Hits         Time  Per Hit   % Time  Line Contents
   305                                                   if batch_len == 0:
   306                                                       return
   307                                           
   308                                                   # Fill buffers with batch data
   309                                                   self.fill_arrays_from_batch(batch)
   310                                           
   311                                                   # Had to do cast stuff to get type checking to play nicely
   312                                                   ens_group = cast(h5py.Group, f["ensembles"])
   313                                           
   314                                                   # Current size of datasets (all should be same length)
   315     